# QBC Small PINN Landscape Study

This notebook analyzes the compact export bundle in `results/pinn_landscape/<experiment_name>/`.

It is designed for the integrated small-QBC PINN landscape study and focuses on:

- training dynamics across `SM4`, `SM6`, and `SM_AVR_GOV`
- early-training behavior
- per-loss contributions to the total loss
- raw and weighted gradient telemetry
- loss landscapes at `init`, midpoint, and end checkpoints


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

RESULTS_ROOT = Path('results') / 'pinn_landscape'
EXPERIMENT_NAME = None  # Set explicitly to override newest-export auto-discovery.
MODEL_ORDER = ['sm4', 'sm6', 'sm_avr_gov']
MODEL_LABELS = {
    'sm4': 'SM4',
    'sm6': 'SM6',
    'sm_avr_gov': 'SM_AVR_GOV',
}
DEFAULT_LOSS_COMPONENTS = ['data', 'dt', 'physics', 'ic']
EARLY_EPOCHS = 50

assert RESULTS_ROOT.exists(), f'Results root not found: {RESULTS_ROOT}'

if EXPERIMENT_NAME is None:
    candidates = [path for path in RESULTS_ROOT.iterdir() if path.is_dir()]
    assert candidates, f'No exported experiments found under: {RESULTS_ROOT}'
    EXPORT_ROOT = max(candidates, key=lambda path: path.stat().st_mtime)
    EXPERIMENT_NAME = EXPORT_ROOT.name
else:
    EXPORT_ROOT = RESULTS_ROOT / EXPERIMENT_NAME

assert EXPORT_ROOT.exists(), f'Export root not found: {EXPORT_ROOT}'
print(f'Using exported experiment: {EXPERIMENT_NAME}')
EXPORT_ROOT


In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text())


def maybe_read_json(path: Path) -> dict | None:
    return read_json(path) if path.is_file() else None


def split_landscape_name(name: str) -> tuple[str, str]:
    if name.endswith('_1d'):
        return name[:-3], '1d'
    if name.endswith('_2d'):
        return name[:-3], '2d'
    return name, 'unknown'


def load_landscape_dir(path: Path) -> dict:
    coords = np.load(path / 'coordinates.npz')
    losses = {}
    for loss_path in sorted(path.glob('loss_*.npy')):
        losses[loss_path.stem.replace('loss_', '')] = np.load(loss_path)
    return {
        'path': path,
        'manifest': maybe_read_json(path / 'manifest.json') or {},
        'coordinates': {key: coords[key] for key in coords.files},
        'losses': losses,
    }


def discover_metric_columns(df: pd.DataFrame) -> dict:
    train_loss_cols = sorted([
        c for c in df.columns
        if c.startswith('train_') and c.endswith('_loss') and c != 'train_total_loss'
    ])
    val_loss_cols = sorted([c for c in df.columns if c.startswith('val_') and c.endswith('_loss')])
    raw_grad_cols = sorted([
        c for c in df.columns
        if c.startswith('train_')
        and c.endswith('_grad_norm')
        and not c.startswith('train_weighted_')
        and c != 'train_total_grad_norm'
    ])
    weighted_grad_cols = sorted([
        c for c in df.columns if c.startswith('train_weighted_') and c.endswith('_grad_norm')
    ])
    test_cols = sorted([c for c in df.columns if c.startswith('test_')])
    return {
        'train_loss_cols': train_loss_cols,
        'val_loss_cols': val_loss_cols,
        'raw_grad_cols': raw_grad_cols,
        'weighted_grad_cols': weighted_grad_cols,
        'test_cols': test_cols,
    }


def load_model_bundle(model_key: str) -> dict:
    model_root = EXPORT_ROOT / model_key
    metrics_path = model_root / 'metrics.csv'
    loss_root = model_root / 'loss_landscape'
    landscapes = {}
    if loss_root.is_dir():
        for path in sorted(loss_root.iterdir()):
            if path.is_dir():
                landscapes[path.name] = load_landscape_dir(path)
    metrics = pd.read_csv(metrics_path) if metrics_path.is_file() else None
    return {
        'model_key': model_key,
        'label': MODEL_LABELS.get(model_key, model_key.upper()),
        'root': model_root,
        'metrics': metrics,
        'metric_columns': discover_metric_columns(metrics) if metrics is not None else {},
        'landscapes': landscapes,
    }


bundles = {
    model_key: load_model_bundle(model_key)
    for model_key in MODEL_ORDER
    if (EXPORT_ROOT / model_key).exists()
}
assert bundles, f'No model bundles found under: {EXPORT_ROOT}'

CHECKPOINT_BASES = sorted({
    split_landscape_name(name)[0]
    for bundle in bundles.values()
    for name in bundle['landscapes']
})
LOSS_NAMES = sorted({
    loss_name
    for bundle in bundles.values()
    for landscape in bundle['landscapes'].values()
    for loss_name in landscape['losses']
}, key=lambda name: (name != 'total', DEFAULT_LOSS_COMPONENTS.index(name) if name in DEFAULT_LOSS_COMPONENTS else 99, name))

print('Models:', list(bundles.keys()))
print('Available checkpoint bases:', CHECKPOINT_BASES)
print('Available loss names:', LOSS_NAMES)


In [ ]:
overview_rows = []
for bundle in bundles.values():
    metrics = bundle['metrics']
    cols = bundle['metric_columns']
    overview_rows.append({
        'model': bundle['label'],
        'epochs_logged': 0 if metrics is None else len(metrics),
        'train_loss_cols': len(cols.get('train_loss_cols', [])),
        'val_loss_cols': len(cols.get('val_loss_cols', [])),
        'raw_grad_cols': len(cols.get('raw_grad_cols', [])),
        'weighted_grad_cols': len(cols.get('weighted_grad_cols', [])),
        'landscape_dirs': len(bundle['landscapes']),
        'checkpoints': ', '.join(sorted({split_landscape_name(name)[0] for name in bundle['landscapes']})),
    })
overview_df = pd.DataFrame(overview_rows)
overview_df


In [ ]:
schema_rows = []
for bundle in bundles.values():
    cols = bundle['metric_columns']
    schema_rows.append({
        'model': bundle['label'],
        'train_losses': ', '.join(cols.get('train_loss_cols', [])),
        'val_losses': ', '.join(cols.get('val_loss_cols', [])),
        'raw_grad_norms': ', '.join(cols.get('raw_grad_cols', [])),
        'weighted_grad_norms': ', '.join(cols.get('weighted_grad_cols', [])),
        'test_metrics': ', '.join(cols.get('test_cols', [])),
    })
pd.DataFrame(schema_rows)


In [ ]:
def _x_values(df: pd.DataFrame) -> np.ndarray:
    return df['global_epoch'].to_numpy() if 'global_epoch' in df.columns else np.arange(1, len(df) + 1)


def _maybe_slice(df: pd.DataFrame, max_epoch: int | None) -> tuple[pd.DataFrame, np.ndarray]:
    x = _x_values(df)
    if max_epoch is None:
        return df.copy(), x
    mask = x <= max_epoch
    return df.loc[mask].reset_index(drop=True), x[mask]


def plot_loss_curves(bundles: dict, max_epoch: int | None = None) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), squeeze=False)
    component_axes = [axes[0, 1], axes[1, 0], axes[1, 1]]
    for bundle in bundles.values():
        if bundle['metrics'] is None:
            continue
        df, x = _maybe_slice(bundle['metrics'], max_epoch)
        axes[0, 0].plot(x, df['train_total_loss'], label=bundle['label'], linewidth=2)
        for ax, component in zip(component_axes, DEFAULT_LOSS_COMPONENTS):
            col = f'train_{component}_loss'
            if col in df.columns:
                ax.plot(x, df[col], label=bundle['label'], linewidth=2)
    axes[0, 0].set_title('Train Total Loss')
    for ax, component in zip(component_axes, DEFAULT_LOSS_COMPONENTS):
        ax.set_title(f'Train {component.capitalize()} Loss')
    for ax in axes.ravel():
        ax.set_xlabel('Global epoch')
        ax.set_yscale('log')
        ax.legend()
    plt.tight_layout()


def plot_eval_curves(bundles: dict, max_epoch: int | None = None) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    any_val = False
    any_test = False
    for bundle in bundles.values():
        if bundle['metrics'] is None:
            continue
        df, x = _maybe_slice(bundle['metrics'], max_epoch)
        if 'val_data_loss' in df.columns:
            axes[0].plot(x, df['val_data_loss'], label=bundle['label'])
            any_val = True
        if 'test_data_loss' in df.columns:
            axes[1].plot(x, df['test_data_loss'], label=bundle['label'])
            any_test = True
    axes[0].set_title('Validation Data Loss')
    axes[1].set_title('Test Data Loss')
    for ax in axes:
        ax.set_xlabel('Global epoch')
        ax.set_yscale('log')
        ax.legend()
    plt.tight_layout()
    if not any_val:
        print('No validation data loss found in metrics.')
    if not any_test:
        print('No test metrics found in metrics.')


def plot_gradient_curves(bundles: dict, weighted: bool, max_epoch: int | None = None) -> None:
    prefix = 'train_weighted_' if weighted else 'train_'
    title_prefix = 'Weighted' if weighted else 'Raw'
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), squeeze=False)
    found_any = False
    for bundle in bundles.values():
        if bundle['metrics'] is None:
            continue
        df, x = _maybe_slice(bundle['metrics'], max_epoch)
        if 'train_total_grad_norm' in df.columns and not df['train_total_grad_norm'].isna().all():
            axes[0, 0].plot(x, df['train_total_grad_norm'], label=bundle['label'], linewidth=2)
            found_any = True
        for ax, component in zip(axes.ravel()[1:], DEFAULT_LOSS_COMPONENTS):
            col = f'{prefix}{component}_grad_norm'
            if col in df.columns and not df[col].isna().all():
                ax.plot(x, df[col], label=bundle['label'], linewidth=2)
                found_any = True
            ax.set_title(f'{title_prefix} {component.capitalize()} Grad Norm')
    axes[0, 0].set_title('Total Grad Norm')
    for ax in axes.ravel():
        ax.set_xlabel('Global epoch')
        ax.set_yscale('log')
        ax.legend()
    plt.tight_layout()
    if not found_any:
        print(f'No {title_prefix.lower()} gradient telemetry columns with values were found.')


plot_loss_curves(bundles)
plot_loss_curves(bundles, max_epoch=EARLY_EPOCHS)
plot_eval_curves(bundles)
plot_gradient_curves(bundles, weighted=False)
plot_gradient_curves(bundles, weighted=True)
plot_gradient_curves(bundles, weighted=False, max_epoch=EARLY_EPOCHS)
plot_gradient_curves(bundles, weighted=True, max_epoch=EARLY_EPOCHS)


In [ ]:
def plot_loss_shares(bundles: dict, max_epoch: int | None = None) -> None:
    fig, axes = plt.subplots(1, len(bundles), figsize=(6 * max(1, len(bundles)), 4), squeeze=False)
    axes = axes[0]
    for ax, bundle in zip(axes, bundles.values()):
        if bundle['metrics'] is None:
            ax.axis('off')
            continue
        df, x = _maybe_slice(bundle['metrics'], max_epoch)
        stacked = []
        for component in DEFAULT_LOSS_COMPONENTS:
            col = f'train_{component}_loss'
            if col in df.columns:
                share = df[col] / df['train_total_loss'].replace(0.0, np.nan)
                stacked.append(np.nan_to_num(share.to_numpy(), nan=0.0, posinf=0.0, neginf=0.0))
            else:
                stacked.append(np.zeros(len(df)))
        ax.stackplot(x, stacked, labels=DEFAULT_LOSS_COMPONENTS, alpha=0.85)
        ax.set_title(f'{bundle["label"]} loss share vs total')
        ax.set_xlabel('Global epoch')
        ax.set_ylabel('share of train_total_loss')
        ax.legend(loc='upper right')
    plt.tight_layout()


def plot_grad_shares(bundles: dict, weighted: bool = True, max_epoch: int | None = None) -> None:
    prefix = 'train_weighted_' if weighted else 'train_'
    label_prefix = 'weighted' if weighted else 'raw'
    fig, axes = plt.subplots(1, len(bundles), figsize=(6 * max(1, len(bundles)), 4), squeeze=False)
    axes = axes[0]
    for ax, bundle in zip(axes, bundles.values()):
        if bundle['metrics'] is None:
            ax.axis('off')
            continue
        df, x = _maybe_slice(bundle['metrics'], max_epoch)
        cols = [
            f'{prefix}{component}_grad_norm'
            for component in DEFAULT_LOSS_COMPONENTS
            if f'{prefix}{component}_grad_norm' in df.columns
        ]
        if not cols:
            ax.set_title(f'{bundle["label"]}: no {label_prefix} grad telemetry')
            ax.axis('off')
            continue
        values = df[cols].fillna(0.0)
        denom = values.sum(axis=1).replace(0.0, np.nan)
        shares = values.divide(denom, axis=0).fillna(0.0)
        ax.stackplot(
            x,
            [shares[col].to_numpy() for col in cols],
            labels=[col.replace(prefix, '').replace('_grad_norm', '') for col in cols],
            alpha=0.85,
        )
        ax.set_title(f'{bundle["label"]} {label_prefix} grad share')
        ax.set_xlabel('Global epoch')
        ax.set_ylabel('share of component grad norm sum')
        ax.set_ylim(0.0, 1.0)
        ax.legend(loc='upper right')
    plt.tight_layout()


plot_loss_shares(bundles)
plot_grad_shares(bundles, weighted=False)
plot_grad_shares(bundles, weighted=True)


In [ ]:
def plot_1d_grid(bundles: dict, checkpoint_bases: list[str] | None = None, loss_names: list[str] | None = None) -> None:
    checkpoint_bases = CHECKPOINT_BASES if checkpoint_bases is None else checkpoint_bases
    loss_names = ['total'] if loss_names is None else loss_names
    for loss_name in loss_names:
        fig, axes = plt.subplots(len(checkpoint_bases), 1, figsize=(10, 4 * max(1, len(checkpoint_bases))), squeeze=False)
        axes = axes[:, 0]
        found_any = False
        for ax, checkpoint_base in zip(axes, checkpoint_bases):
            key = f'{checkpoint_base}_1d'
            for bundle in bundles.values():
                landscape = bundle['landscapes'].get(key)
                if landscape is None or loss_name not in landscape['losses']:
                    continue
                alpha = landscape['coordinates']['alpha']
                values = landscape['losses'][loss_name]
                ax.plot(alpha, values, marker='o', label=bundle['label'])
                found_any = True
            ax.set_title(f'1D landscape | checkpoint={checkpoint_base} | loss={loss_name}')
            ax.set_xlabel('alpha')
            ax.set_ylabel('loss')
            ax.set_yscale('log')
            ax.legend()
        plt.tight_layout()
        if not found_any:
            print(f'No 1D landscapes found for loss={loss_name!r}.')


def plot_2d_comparison(bundles: dict, checkpoint_base: str, loss_name: str = 'total') -> None:
    key = f'{checkpoint_base}_2d'
    available = [(name, bundle) for name, bundle in bundles.items() if key in bundle['landscapes']]
    if not available:
        print(f'No 2D landscapes found for checkpoint={checkpoint_base!r}.')
        return
    fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 5), squeeze=False)
    axes = axes[0]
    for ax, (_, bundle) in zip(axes, available):
        landscape = bundle['landscapes'][key]
        if loss_name not in landscape['losses']:
            ax.axis('off')
            continue
        alpha = landscape['coordinates']['alpha']
        beta = landscape['coordinates']['beta']
        values = landscape['losses'][loss_name]
        contour = ax.contourf(alpha, beta, np.log10(values + 1e-16).T, levels=25, cmap='viridis')
        ax.set_title(bundle['label'])
        ax.set_xlabel('alpha')
        ax.set_ylabel('beta')
        fig.colorbar(contour, ax=ax, label=f'log10({loss_name})')
    fig.suptitle(f'2D landscape comparison | checkpoint={checkpoint_base} | loss={loss_name}', y=1.02)
    plt.tight_layout()


plot_1d_grid(bundles, checkpoint_bases=CHECKPOINT_BASES, loss_names=['total'])
plot_1d_grid(bundles, checkpoint_bases=CHECKPOINT_BASES, loss_names=[name for name in LOSS_NAMES if name != 'total'])
for checkpoint_base in CHECKPOINT_BASES:
    plot_2d_comparison(bundles, checkpoint_base=checkpoint_base, loss_name='total')
for checkpoint_base in CHECKPOINT_BASES:
    for loss_name in [name for name in LOSS_NAMES if name != 'total']:
        plot_2d_comparison(bundles, checkpoint_base=checkpoint_base, loss_name=loss_name)


In [ ]:
training_rows = []
for bundle in bundles.values():
    metrics = bundle['metrics']
    if metrics is None or metrics.empty:
        continue
    first = metrics.iloc[0]
    mid = metrics.iloc[len(metrics) // 2]
    last = metrics.iloc[-1]
    row = {'model': bundle['label']}
    for prefix, src in [('start', first), ('mid', mid), ('end', last)]:
        row[f'{prefix}_epoch'] = int(src.get('global_epoch', src.name + 1))
        for name in ['train_total_loss'] + [f'train_{component}_loss' for component in DEFAULT_LOSS_COMPONENTS]:
            if name in metrics.columns:
                row[f'{prefix}_{name}'] = float(src[name])
        for name in ['train_total_grad_norm'] + [f'train_{component}_grad_norm' for component in DEFAULT_LOSS_COMPONENTS] + [f'train_weighted_{component}_grad_norm' for component in DEFAULT_LOSS_COMPONENTS]:
            if name in metrics.columns and not pd.isna(src[name]):
                row[f'{prefix}_{name}'] = float(src[name])
    training_rows.append(row)
training_summary_df = pd.DataFrame(training_rows)
training_summary_df


In [ ]:
landscape_rows = []
for bundle in bundles.values():
    for landscape_name, landscape in bundle['landscapes'].items():
        checkpoint_base, grid = split_landscape_name(landscape_name)
        for loss_name, values in landscape['losses'].items():
            center_idx = tuple(dim // 2 for dim in values.shape)
            landscape_rows.append({
                'model': bundle['label'],
                'checkpoint': checkpoint_base,
                'grid': grid,
                'loss_name': loss_name,
                'center_loss': float(values[center_idx]),
                'min_loss': float(np.nanmin(values)),
                'max_loss': float(np.nanmax(values)),
                'spread': float(np.nanmax(values) - np.nanmin(values)),
            })
landscape_summary_df = pd.DataFrame(landscape_rows).sort_values(['grid', 'checkpoint', 'loss_name', 'model']).reset_index(drop=True)
landscape_summary_df
